# 🔬 Benchmarking Search Retrieval with **Azure AI Foundry**

Welcome to Lab 📒 **03 — Search Evaluation** in our Gen-AI search series!  
In this notebook we’ll turn raw **ranking results** from the previous experiment into actionable **NDCG** and **Recall** scores using Azure AI Foundry’s evaluation service.

---

## 🌟 Why run this lab?

| ✅ Outcome | 🔎 Detail |
|-----------|-----------|
| **Per-query insight** | Compute NDCG@3, NDCG@10, and Recall@10 for every query. |
| **Aggregate leaderboard** | Foundry auto-aggregates scores so you can crown the best retrieval method. |
| **Reproducibility** | Git hash + custom tags ensure every run is traceable. |
| **One-click compare** | Flip through metrics across *hybrid-semantic*, *hybrid*, *keyword*, and *vector* pipelines in the Foundry UI. |

## Getting Started

First thing we're going to do is setup our environment with the connectivity to AI Foundry by importing all of the required files. We're also going to record our current git state so we understand the version we're working on for future tracking as well.

> The SDK for AI Foundry v1.2.0 doesn't support custom tagging, so the `custom_start_run` method is provided to help facilitate a passthrough for creating custom tags with the evaluation SDK.

In [3]:
import sys
import os
import json
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import evaluate
from azure.ai.evaluation._evaluate._eval_run import EvalRun

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.evals.custom.ndcg_evaluator import NDCGEvaluator
from src.evals.custom.search_recall_evaluator import SearchRecallEvaluator
import src.evals.sdk.custom_azure_ai_evaluations as custom_eval

load_dotenv()
EvalRun._start_run = custom_eval.custom_start_run

conn_str = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
_, AZ_SUBSCRIPTION_ID, AZ_RESOURCE_GROUP, AI_FOUNDRY_PROJECT_NAME = conn_str.split(";")
azure_ai_project = {
    "subscription_id": AZ_SUBSCRIPTION_ID,
    "resource_group_name": AZ_RESOURCE_GROUP,
    "project_name": AI_FOUNDRY_PROJECT_NAME,
}
credential = DefaultAzureCredential()
ai_project_client = AIProjectClient.from_connection_string(credential=credential, conn_str=conn_str)
print(f"Connected to Azure AI Project: {AI_FOUNDRY_PROJECT_NAME}")

git_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"🔖 Using commit: {git_hash}")

## Loading our Benchmark Dataset for Search Retrievals 📦🔍  

Search pipelines aren’t “set-and-forget” — they’re living systems whose quality can drift whenever you tweak an embedding model, ingest fresh documents, or enable a new semantic re-ranker. Even tiny changes can move your NDCG or Recall needle by double-digit percentages, which ultimately affects end-user trust and downstream Gen-AI chains. Azure AI Foundry gives you an industrial-grade way to **measure, compare, and monitor** those shifts so you can act before they hurt production. Below we expand on *why* Foundry’s evaluation tooling is essential and how it fits into your MedIndexer lab workflow.  

### ⚙️ Why continuous search benchmarking matters  
- **Model volatility** – Upgrading the embedding model or its hyper-parameters can reorder vectors and disturb similarity scores, instantly changing result sets.  
- **Index churn** – Adding or re-indexing content reshapes term statistics and vector densities, so precision/recall must be re-validated.  
- **Ranking features** – Enabling semantic ranker or query-rewrite can boost relevance, but the gains vary by corpus and must be quantified (e.g., +4 NDCG@3 reported when QR was introduced).  
- **Hidden regressions** – Schema or prompt tweaks in a RAG pipeline may silently degrade answer accuracy unless metrics flag the drop.  
- **Production drift** – Live user queries evolve; without ongoing measurement you won’t notice that yesterday’s top-ranker is today’s under-performer.  

### 🏭 How AI Foundry solves it  
| Capability | Why it helps for search | Foundry feature |
|------------|------------------------|-----------------|
| **Automated metrics** | Built-in evaluators compute NDCG, Recall & custom retrieval scores on each run. | `RetrievalEvaluator`, `RelevanceEvaluator` |
| **Leaderboards** | Side-by-side charts let you crown the best pipeline (keyword vs vector vs hybrid-semantic). | Model & pipeline leaderboards |
| **Versioned experiments** | Git hash + tags lock every run to code/data so you can reproduce or roll back. | Run metadata & tagging API |
| **Online evaluation** | Schedule continuous checks that write results to App Insights for alerting. | Foundry Online Evaluation + Azure Monitor |
| **Cross-scenario comparability** | Same framework scores chat response quality, safety, and search in one place for holistic app health. | Unified evaluator catalog |

### 🛠️ Expanded preprocessing workflow for MedIndexer  
1. **Generate query-ranking pairs** from your *aihlsignited-medindexer* notebook, storing top-k results for each synthetic or real query.  
2. **Label or derive ground-truth** relevance judgments (LLM-assisted or human).  
3. **Package as a Foundry Dataset** so each experiment consumes the exact same benchmark slice.  
4. **Tag the evaluation run** with `search-retrieval-performance`, pipeline type (`hybrid`, `vector`, etc.), and Git commit for traceability.  
5. **Submit an Evaluation job** that calculates NDCG@3/10, Recall@10, and any custom metrics you registered.  
6. **Inspect leaderboards & trend lines** to verify improvements or catch regressions before redeploying.  

### 🚦 What you gain  
- **Objective guard-rails** – Decisions are driven by metrics, not anecdotes.  
- **Rapid iteration** – Inline dashboards surface winners immediately, shortening your experiment cycle.  
- **Regulatory readiness** – Persistent audit logs satisfy evidence requirements for healthcare AI systems.  
- **Team alignment** – Shared leaderboards keep data scientists, ops, and clinicians on the same page about “what’s best.”  
- **Peace of mind** – Continuous evaluation plus Azure Monitor alerts let you sleep at night knowing search quality is watched 24/7.  

> **TL;DR** 🗒️  
> Azure AI Foundry turns one-off relevance checks into a repeatable, version-controlled pipeline with live monitoring. That’s the foundation you need to keep your MedIndexer search experience accurate, safe, and future-proof as models, data, and user behavior inevitably evolve.  

---

### 🧹 Preprocessing & Dataset Generation (the *must-do* step before evaluation)  

Creating a high-fidelity benchmark dataset is the linchpin of trustworthy metrics: if the data is noisy or incomplete, the scores you compute in Foundry will be meaningless. Preprocessing transforms raw retrieval outputs into a structured **`.jsonl`** file that every Foundry evaluator understands. 

It's a process that is consistent to each and every evaluation.

**Why preprocess?**  
- Evaluators require specific fields (`query`, `response`, `context`, etc.); missing keys will cause validation failures.  [Evaluation SDK](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/evaluate-sdk)  
- Aligning each query with a deterministic *ground truth* snapshot ensures that later index changes don’t retro-edit historic labels.  [Microsoft Evaluation Approach](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-approach-gen-ai)  

**End-to-end preprocessing checklist**  
1. **Collect representative queries** — harvest from production logs or synthetically generate via LLM prompts that mirror user intent.  [Simulation Generator with Microsoft](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/simulator-interaction-data)  
2. **Run your retrieval pipeline** to capture top-k passages plus the model-generated answer (if applicable).  
3. **Attach ground-truth labels** — manual annotation or heuristics such as document-id matches for closed-book QA, if needed for your evaluators.
4. **Serialize to `.jsonl`** with required keys and optional metadata:  
   ```jsonl
    {   
        ...
        "query": "Crohn's disease Adalimumab prior authorization criteria", 
        "response": {"ranking": {"d40": 2.8734118938446045, ... "d42": 1.0734176635742188}}, 
        "ground_truth": {"d4": 0, ... "d71": 1}
        ...
    }

   ```  
   Each line is an independent evaluation example. 

Following this pipeline guarantees that every Evaluation run is apples-to-apples, reproducible, and actionable — the holy trinity of production-grade evaluation.

In [ ]:
EVAL_DIR = Path("../evals/benchmark/medindexer")
queries_path = EVAL_DIR / "queries.jsonl"
qrels_path   = EVAL_DIR / "qrels.jsonl"
ranking_files = {
    "hybrid_semantic": EVAL_DIR / "rankings-hybrid-semantic.jsonl",
    "hybrid":          EVAL_DIR / "rankings-hybrid.jsonl",
    "keyword":         EVAL_DIR / "rankings-keyword.jsonl",
    "vector":          EVAL_DIR / "rankings-vector.jsonl",
}

queries = {}
with open(queries_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        queries[e["id"]] = e["query"]

qrels = {}
with open(qrels_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        qrels.setdefault(e["query"], {})[e["document"]] = int(e["relevant"])

for method, file_path in ranking_files.items():
    out_path = EVAL_DIR / f"eval_data_{method}.jsonl"
    with open(file_path, "r", encoding="utf-8") as fin, \
         open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)
            qid = rec["query"]
            # sort docs by score desc
            sorted_docs = [
                doc for doc, _ in sorted(rec["ranking"].items(), key=lambda x: x[1], reverse=True)
            ]
            record = {
                "query_id":     qid,
                "query":        queries.get(qid, ""),
                "method":       method,
                # embed the ranking under "context"
                "response":      { "ranking": rec["ranking"] },
                # emit ground_truth as a JSON object
                "ground_truth": qrels.get(qid, {}),
            }
            fout.write(json.dumps(record) + "\n")
    print(f"📦 Preprocessed data for '{method}' → {out_path}")


## Evaluation Phase 🚀🧮  

In this stage we turn the *rankings generated during preprocessing* into hard numbers—**NDCG** and **Search Recall**—so we can know with confidence whether a new embedding model, index rebuild, or semantic re-ranker actually helps users. We’ll run the evaluation locally (great for rapid prototyping and CI jobs) but the exact same script can execute on a remote agent wired into your production pipeline, keeping your Foundry leaderboards and alerts fully up to date.  

---

### 🔑 Why we evaluate  
- **Objective quality control** – Metrics like Normalized Discounted Cumulative Gain measure how well highly-relevant docs are surfaced near the top of the list, mirroring real user satisfaction.  
- **Recall assurance** – Recall@ k checks that crucial documents are **never** lost after a model or index tweak, a must-have in regulated domains.  
- **CI/CD fit** – Automation means every pull request is scored, preventing silent regressions before they hit production search.  
- **Foundry traceability** – By attaching run-level tags (e.g., `commit=a1b2c3d`, `pipeline=hybrid`) we lock every score to the exact code and data that produced it. 

---

### 📐 Metrics in focus  

**NDCG@k** weighs ranked positions so that highly-relevant docs scored at rank 1 are worth more than the same docs at rank 5, giving a user-centric “gain” curve.  

**Recall@k** answers a binary question: did *all* the ground-truth docs appear in the top *k*? It guards against catastrophic misses when new data is ingested. 

Both metrics are part of the BEIR toolkit, which standardises evaluation across 20+ IR datasets.  

---

### 🧩 Custom evaluators & tagging  

Foundry ships built-ins, but you can plug in anything callable. Below we wrap BEIR’s `EvaluateRetrieval` so it plays nicely with Foundry’s evaluator interface and returns a dict that auto-populates metric dashboards.  

```python
from beir.retrieval.evaluation import EvaluateRetrieval  # ⛽ BEIR core
class NDCGEvaluator:
    def __init__(self, k: int = 10):
        self.k = k

    def __call__(self, *, context: dict, ground_truth: dict, query_id: str, **kwargs):
        ranking   = context.get("ranking", {})          # {doc_id: score}
        qrels     = {query_id: ground_truth}            # {query_id: {doc_id: rel}}
        results   = EvaluateRetrieval().evaluate(
                        qrels, {query_id: ranking}, k_values=[self.k]
                    )
        return {f"NDCG@{self.k}": results[0].get(f"NDCG@{self.k}", 0.0)}
```

> **Tag helper** – wrap the evaluator call in a small utility that injects `("commit", git_hash)` and `("pipeline", pipeline_type)` so every run shows up grouped in Foundry leaderboards.  [oai_citation:15‡Microsoft Learn](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/data-add?utm_source=chatgpt.com)  

---

### 🏃‍♀️ Running the evaluation & storing results  

1. **Load the preprocessed JSONL** created earlier (one line per query-response pair).  
2. **Instantiate evaluators** – e.g., `ndcg10 = NDCGEvaluator(k=10)` plus Foundry’s built-in `RecallEvaluator`.  
3. **Call `evaluate()`** from the `azure.ai.evaluation` SDK, passing the dataset path, evaluators dict, and tag list.   
4. **Persist** the resulting `evaluation_results.jsonl` so CI can publish an artifact and Foundry can ingest it for long-term trend charts. 


In [9]:
def _generate_custom_tags(case_id: str, git_commit: str, class_name: str):
    tags = [
        ("case", case_id),
        ("commit", git_commit),
        ("embeddings", "text-embedding-3-large"),
    ]
    custom_eval.CUSTOM_TAGS = tags
    return tags

results = []
for method in ranking_files:
    eval_file = EVAL_DIR / f"eval_data_{method}.jsonl"
    eval_name = f"search-retrieval-{method}-{git_hash}"

    _generate_custom_tags(case_id=method, git_commit=git_hash, class_name="SearchRetrievalEvaluator")

    result = evaluate(
        evaluation_name=eval_name,
        data=str(eval_file),
        evaluators={
            "ndcg_3":    NDCGEvaluator(3),
            "ndcg_10":   NDCGEvaluator(10),
            "recall_10": SearchRecallEvaluator(10),
        },
        azure_ai_project=azure_ai_project,
        evaluator_config={
            metric: {
                "column_mapping": {
                    # map the full context object
                    "response":      "${data.response}",
                    "ground_truth": "${data.ground_truth}",
                    "query_id":     "${data.query_id}"
                }
            }
            for metric in ["ndcg_3", "ndcg_10", "recall_10"]
        }
    )
    results.append(result)
    print(f"✅ Completed '{method}':", result["metrics"])

[2025-04-26 17:56:57 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-26 17:56:57 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-26 17:56:57 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-26 17:56:57 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_c7lv8afj_20250426_175657_907887, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_c7lv8afj_20250426_175657_907887/logs.txt
[2025-04-26 17:56:57 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_

Although the automated pipeline is , we can begin to outline the next steps to enable future **post-processing** steps.

1. **Plan evaluation visualisations** – define which charts (NDCG @ k, Recall @ k, precision curves, etc.) should be generated automatically after each run.  These can be artifacts included as part of your final release artifact.
2. **Integrate with release management** – map evaluation pass/fail thresholds to CI/CD gates so only models meeting target metrics progress to staging or production.
3. **Custom formatting as required** - you can format the results as needed for any customization you require, so that you further integrate the evaluations as you need

Happy evaluating! More on to the next notebook as we finalize the building of pipelines.